## Cargar TinyLlama con Unsloth

In [1]:
# Verificar si hay GPU disponible
import torch

print("PyTorch version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))
    print("Número de GPUs:", torch.cuda.device_count())
    print("Memoria GPU total:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")
else:
    print("⚠️ No se detectó GPU CUDA")
    print("Verifica que:")
    print("  1. Tienes una GPU NVIDIA instalada")
    print("  2. Tienes los drivers NVIDIA actualizados")
    print("  3. Tienes CUDA Toolkit instalado")
    print("  4. Tienes la versión correcta de PyTorch con soporte CUDA")

PyTorch version: 2.9.1+cpu
CUDA disponible: False
⚠️ No se detectó GPU CUDA
Verifica que:
  1. Tienes una GPU NVIDIA instalada
  2. Tienes los drivers NVIDIA actualizados
  3. Tienes CUDA Toolkit instalado
  4. Tienes la versión correcta de PyTorch con soporte CUDA


In [1]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,   # QLoRA
)


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

Aquí Unsloth:

- Descarga el modelo desde Hugging Face
- Lo cuantiza a 4-bit
- Lo deja listo para LoRA

## Inyectar LoRA (ajustes recomendados para TinyLlama)

TinyLlama es pequeño, no conviene usar LoRA grande.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,   # 👈 menor rango (clave)
    target_modules = [
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = True,
)


Justificación técnica

- r=8 evita overfitting
- TinyLlama no tolera LoRA grandes
- Ideal para datasets médicos pequeños